In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from ic_benchmark_suite import repetition_decrease, idyom_corr
from ic_benchmark_suite.dataset_provider import DatasetProvider
from ic_benchmark_suite.baseline.audioic import AudioIC

/home/mathias/.conda/envs/ic_benchmark_suite/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load dataset provider for accsessing datasets
dataset_provider = DatasetProvider(
    cache_dir="./cache",
    hf_root="muthissar/music_surprisal_benchmarks",
    hf_token=os.environ.get("HF_ACCESS_TOKEN"),
)

# Pitch Surprisal - IDyOM consistency

In [4]:
# Get session paired audio, idyom dataset from the hub
session_ds = dataset_provider.get_ds("session_ds")
session_ds.features

Using the latest cached version of the dataset since muthissar/music_surprisal_benchmarks couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'session_ds' at cache/muthissar___music_surprisal_benchmarks/session_ds/0.0.0/c036686ccc9f24e42c41951eea9b6926a51dd3d9 (last modified on Fri Mar 27 13:54:23 2026).


{'id': Value('int64'),
 'onset': List(Value('float64')),
 'audios': {'Acoustic Grand Piano': Value('binary'),
  'Acoustic Guitar (nylon)': Value('binary'),
  'Pad 1 (new age)': Value('binary'),
  'Synth Voice': Value('binary'),
  'Trumpet': Value('binary')},
 'idyom_ic': List(Value('float64')),
 'idyom_entropy': List(Value('float64'))}

In [13]:
# optionally filter dataset (for faster testing)
n_audios = 20
session_ds_small = session_ds.select(range(n_audios))

In [14]:
# load baseline model
baseline = AudioIC(device='cpu')

In [15]:
ic_ds = baseline.ic(session_ds_small)

Computing IC: 100%|██████████| 20/20 [05:24<00:00, 16.23s/it]


In [16]:
ic_ds.features

{'id': Value('int64'),
 'instrument': Value('string'),
 'ic': List(Value('float32'))}

In [18]:
# load only the relevant columns for idyom IC correlation from session_ds (idyom_ic)
idyom_ds = session_ds_small.select_columns(["id", "onset",  "idyom_ic"]) 
idyom_ds = idyom_ds.rename_column("idyom_ic", "idyom_val")
result = idyom_corr(val_dataset=ic_ds, idyom_ds=idyom_ds, periode_frames=AudioIC.periode_frames())
result

[{'instrument': 'combined',
  'label': 'ic',
  'r_val': np.float64(0.5182819160179176),
  'p_val': np.float64(0.0)}]

In [31]:
ic_ds

Dataset({
    features: ['id', 'instrument', 'ic'],
    num_rows: 1000
})

In [ ]:
# running the experiment on all values in the dataset
ic_ds = dataset_provider.get_ds("cache/ic/audioic_session_ic")
idyom_ds = session_ds.select_columns(["id", "onset",  "idyom_ic"]) 
idyom_ds = idyom_ds.rename_column("idyom_ic", "idyom_val")
result = idyom_corr(val_dataset=ic_ds, idyom_ds=idyom_ds, periode_frames=AudioIC.periode_frames())
result

Using the latest cached version of the dataset since muthissar/music_surprisal_benchmarks couldn't be found on the Hugging Face Hub


[{'instrument': 'combined',
  'label': 'ic',
  'r_val': np.float64(0.4704113919798129),
  'p_val': np.float64(0.0)}]

# Repetition lowers IC

In [ ]:

# Get precomputed IC values for salami dataset
model_outputs = dataset_provider.get_ds("audioic_salami_ic")
# Get salami segment annotation metadata from local disk
metadata_ds = dataset_provider.get_ds("./cache/datasets/salami_metadata_ds")
results = repetition_decrease(
    model_outputs=model_outputs,
    metadata_ds=metadata_ds,
    periode_frames = AudioIC.periode_frames(),
    time_offset_frame =  1 # for IC 1 for entropy 0
)

In [ ]:
print({label: {annotation: res[annotation][0] for annotation in res} for label, res in results.items()})